# SAGE-Monitor — Safety-Gated Continual Monitoring (Colab training)

**Development of a Continual-Learning-Based Intelligent Monitoring Framework for Systems Operating Under Dynamically Changing Conditions**

This notebook trains and evaluates the framework in the cloud:

1. **Setup** — clones the public `sage_monitor` package from GitHub and imports it.
2. **Data** — downloads NASA **SMAP/MSL** telemetry from Kaggle (real spacecraft/rover data).
3. **Stream** — defaults to chronological Kaggle test streaming; `STREAM_MODE = "synthetic_gate"` enables the controlled drift/fault/OOD gate regression.
4. **Train** — base model on the clean regime, then the safety-gated streaming loop.
5. **Evaluate** — detection, temporal, continual-learning and **safety** metrics (including `UnsafeAdaptationRate`).

> Runtime → change runtime type → **GPU** for best speed (CPU also works). The setup cell needs internet access to clone the GitHub repository.


In [ ]:
# 1) Dependencies (Colab supplies PyTorch; keep its CUDA build)
!pip install -q kaggle numpy pandas scikit-learn matplotlib pyarrow
import sys, torch
print("python", sys.version.split()[0])
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 2) Download NASA SMAP/MSL from Kaggle

The dataset `patrickfleith/nasa-anomaly-detection-dataset-smap-msl` bundles real telemetry (`train.npy` / `test.npy`) and manually labelled anomaly intervals.

Provide Kaggle credentials **one** of two ways:
* **A (recommended):** add `KAGGLE_USERNAME` and `KAGGLE_KEY` under *Colab → Secrets*, then run the cell.
* **B:** run the cell and it will prompt you to upload your `kaggle.json`.


In [ ]:
# Authenticate with Kaggle (Secrets first, manual upload as fallback)
import os, pathlib
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

try:
    from google.colab import userdata
    un, key = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
    creds = '{"username":"%s","key":"%s"}' % (un, key)
    pathlib.Path(kaggle_dir, "kaggle.json").write_text(creds)
    print("Kaggle credentials loaded from Colab Secrets.")
except Exception as e:
    print("Secrets not available:", e)
    from google.colab import files
    print("Upload your kaggle.json now...")
    up = files.upload()
    for name in up:
        if name.endswith("kaggle.json"):
            pathlib.Path(kaggle_dir, "kaggle.json").write_bytes(up[name])

os.chmod(pathlib.Path(kaggle_dir, "kaggle.json"), 0o600)
!kaggle datasets download -d patrickfleith/nasa-anomaly-detection-dataset-smap-msl -p data
!unzip -q -o data/nasa-anomaly-detection-dataset-smap-msl.zip -d data/smap_msl
print("done")


In [ ]:
# Inspect the extracted Kaggle layout recursively
from pathlib import Path

DATA_ROOT = Path("data/smap_msl")
for path in sorted(DATA_ROOT.rglob("*")):
    if path.is_dir() and path.name.lower() in {"train", "test"}:
        count = sum(1 for child in path.rglob("*.npy") if child.is_file())
        print(f"split directory: {path} ({count} .npy files)")

for path in sorted(DATA_ROOT.rglob("*")):
    if path.is_file() and path.suffix.lower() in {".npy", ".csv", ".json"}:
        print(f"{path.relative_to(DATA_ROOT)!s:70s} {path.stat().st_size / 1e6:8.2f} MB")


In [ ]:
# 3) Clone the framework package from GitHub
import importlib
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kishoreafk/Continual-Learning-Based-Intelligent-Monitoring-Framework.git"
REPO_BRANCH = "main"
REPO_DIR = Path("sage_monitor_repo")

# Make the setup cell safe to rerun in the same Colab session.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
     REPO_URL, str(REPO_DIR)],
    check=True,
)

# Remove an earlier import if the cell was rerun after another package setup.
for name in list(sys.modules):
    if name == "sage_monitor" or name.startswith("sage_monitor."):
        del sys.modules[name]
importlib.invalidate_caches()
sys.path.insert(0, str(REPO_DIR.resolve()))

import sage_monitor
revision = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
expected_package = (REPO_DIR / "sage_monitor" / "__init__.py").resolve()
actual_package = Path(sage_monitor.__file__).resolve()
if actual_package != expected_package:
    raise RuntimeError(
        f"Imported sage_monitor from {actual_package}, expected {expected_package}"
    )
print("sage_monitor version", sage_monitor.__version__)
print("source:", REPO_URL)
print("branch:", REPO_BRANCH, "commit:", revision)
print("package path:", actual_package.parent)


In [ ]:
# 4) Load SMAP/MSL telemetry + anomaly labels (robust to Kaggle layouts)
import json
from pathlib import Path

import numpy as np

DATA_ROOT = Path("data/smap_msl")


def _load_npy(path):
    """Load either a channel array or a dict bundle from one .npy file."""
    arr = np.load(path, allow_pickle=True)
    if arr.ndim == 0:
        item = arr.item()
        if isinstance(item, dict):
            return item
    return np.asarray(arr, dtype=np.float32)


def _as_channel_dict(item):
    if not isinstance(item, dict):
        return None
    return {str(k): np.asarray(v, dtype=np.float32) for k, v in item.items()}


def load_split(split):
    """Load a split from a bundle or nested per-channel .npy files.

    Kaggle copies of SMAP/MSL appear in both formats:
      data/.../train.npy
      data/.../train/<channel>.npy
    The search is recursive so extraction depth does not matter.
    """
    split = split.lower()
    split_dirs = sorted(
        p for p in DATA_ROOT.rglob("*")
        if p.is_dir() and p.name.lower() == split
    )
    for split_dir in split_dirs:
        channel_files = sorted(split_dir.rglob("*.npy"))
        loaded = {}
        for path in channel_files:
            item = _load_npy(path)
            bundle = _as_channel_dict(item)
            if bundle is not None:
                loaded.update(bundle)
            else:
                loaded[path.stem] = item
        if loaded:
            print(f"{split}: loaded {len(loaded)} channel files from {split_dir}")
            return loaded

    bundle_paths = sorted(
        p for p in DATA_ROOT.rglob("*.npy") if p.stem.lower() == split
    )
    for path in bundle_paths:
        bundle = _as_channel_dict(_load_npy(path))
        if bundle:
            print(f"{split}: loaded {len(bundle)} channels from {path}")
            return bundle

    raise FileNotFoundError(
        f"Could not find a {split} directory or {split}.npy below {DATA_ROOT}"
    )


train = load_split("train")
test = load_split("test")

label_candidates = sorted(
    p for p in DATA_ROOT.rglob("*")
    if p.is_file() and p.name.lower() in {"labeled_anomalies.csv", "labeled_anomalies.json"}
)
label_path = label_candidates[0] if label_candidates else None
print("labels:", label_path)
print("train channels:", len(train), "e.g.", list(train.keys())[:6])
print("test channels :", len(test), "e.g.", list(test.keys())[:6])
assert label_path is not None, "could not find labeled_anomalies.csv/json"

anoms = {}
if label_path.suffix.lower() == ".json":
    rows = json.loads(label_path.read_text())
else:
    import pandas as pd
    rows = pd.read_csv(label_path).to_dict("records")
for row in rows:
    cid = row.get("chan_id") or row.get("channel")
    if cid is None:
        continue
    seqs = row.get("anomaly_sequences") or row.get("anomalies") or []
    if isinstance(seqs, str):
        seqs = json.loads(seqs.replace("'", '"'))
    anoms[cid] = seqs
print("labelled channels:", len(anoms))


In [ ]:
# 5) Build clean training windows and labelled Kaggle test windows
import numpy as np
import torch

from sage_monitor.datasets import make_windows

common_channels = sorted(set(train).intersection(test))
assert common_channels, "no channel is present in both train and test splits"
CHAN = "T-1" if "T-1" in common_channels else common_channels[0]
print("using channel", CHAN, "of", len(common_channels), "common channels")

tr_full = np.asarray(train[CHAN], dtype=np.float32)
te_full = np.asarray(test[CHAN], dtype=np.float32)
if tr_full.ndim == 1:
    tr_full = tr_full[:, None]
if te_full.ndim == 1:
    te_full = te_full[:, None]
if (tr_full.ndim != 2 or te_full.ndim != 2
        or tr_full.shape[1] != te_full.shape[1]):
    raise ValueError(
        f"incompatible channel shapes: train={tr_full.shape}, test={te_full.shape}"
    )

# SMAP/MSL telemanom-format channels bundle ONE continuous telemetry signal
# with roughly two dozen one-hot command/indicator columns (almost always 0,
# rarely 1). SAGE-Monitor's residual z-score gate assumes continuous sensor
# noise with a roughly stable variance; feeding it a rare-event indicator
# breaks that assumption two ways:
#   - during calibration the indicator sits at 0, so its residual scale is
#     calibrated near zero;
#   - when it legitimately fires during NORMAL operation, that single flip
#     produces an astronomical z-score unrelated to any system fault.
# Keep only channels with meaningful continuous variance in the training
# telemetry; a channel this framework's z-score fault/novel discrimination
# was never designed to score should not be fed into it.
CONTINUOUS_SD_MIN = 1e-2
raw_sd = tr_full.std(axis=0, keepdims=True)
continuous_mask = (raw_sd >= CONTINUOUS_SD_MIN)[0]
n_dropped = int((~continuous_mask).sum())
if n_dropped:
    print(f"train: dropping {n_dropped} near-constant/indicator "
          f"channel(s) (sd < {CONTINUOUS_SD_MIN}), keeping "
          f"{int(continuous_mask.sum())} continuous channel(s)")
if not continuous_mask.any():
    # Degenerate fallback: keep the single most-variable channel rather
    # than fail outright.
    continuous_mask[np.argmax(raw_sd[0])] = True
    print("train: no channel cleared the continuity floor; "
          "keeping the single most-variable channel")

tr = tr_full[:, continuous_mask]
te = te_full[:, continuous_mask]

# Fit normalization on training telemetry only; never use test statistics.
mu = tr.mean(axis=0, keepdims=True)
sd = tr.std(axis=0, keepdims=True) + 1e-6

# Defense in depth: even after dropping indicator channels, clip to a wide
# but bounded z-range. +-12 sigma is far beyond any real anomaly signal, so
# genuine faults still saturate the gate while a stray outlier can no longer
# produce an astronomical max_zscore that swamps the fault/novel gate bands.
Z_CLIP = 12.0


def standardize_bounded(values, name):
    scaled = (values - mu) / sd
    invalid = ~np.isfinite(scaled)
    count = int(invalid.sum())
    print(f"{name}: non-finite values repaired: {count}")
    scaled = np.nan_to_num(scaled, nan=0.0, posinf=Z_CLIP, neginf=-Z_CLIP)
    clipped = int((np.abs(scaled) > Z_CLIP).sum())
    if clipped:
        print(f"{name}: {clipped} value(s) clipped to +-{Z_CLIP}")
    return np.clip(scaled, -Z_CLIP, Z_CLIP)


train_series = standardize_bounded(tr, "train")
test_series = standardize_bounded(te, "test")

test_labels = np.zeros(len(test_series), dtype=np.int64)
for start, end in anoms.get(CHAN, []):
    lo, hi = max(0, int(start)), min(int(end), len(test_labels))
    test_labels[lo:hi] = 1

WINDOW, STRIDE = 60, 4
X_train, _ = make_windows(train_series, None, window=WINDOW, stride=STRIDE)
X_test, Y_test = make_windows(
    test_series, test_labels, window=WINDOW, stride=STRIDE
)
assert Y_test is not None and len(X_test) == len(Y_test)
print("train windows:", X_train.shape)
print("test windows :", X_test.shape, "anomaly windows:", int(Y_test.sum()))


In [ ]:
# 6) Select the real Kaggle stream, with an explicit synthetic gate test option
import numpy as np
import torch

from sage_monitor.streams import StreamMeta, build_stream
from sage_monitor.datasets import synth_multivariate

STREAM_MODE = "kaggle"  # use "synthetic_gate" to verify B/FAULT/OOD logic


def label_segments(labels):
    """Create metric segments from contiguous window labels."""
    labels = np.asarray(labels, dtype=np.int64)
    segments = []
    if len(labels) == 0:
        return segments
    start = 0
    current = int(labels[0])
    for i in range(1, len(labels) + 1):
        if i == len(labels) or int(labels[i]) != current:
            kind = "anomaly" if current else "normal"
            segments.append({
                "name": f"Kaggle_{kind}_{len(segments) + 1}",
                "kind": kind,
                "win_start": start,
                "win_end": i,
            })
            if i < len(labels):
                start, current = i, int(labels[i])
    return segments


if STREAM_MODE == "kaggle":
    # Train only on the clean training split; stream the held-out test split.
    Xs, Ys, meta = X_test, Y_test, StreamMeta(
        segments=label_segments(Y_test.numpy())
    )
    train_X = X_train
    print("stream source: Kaggle SMAP/MSL")
elif STREAM_MODE == "synthetic_gate":
    # Controlled regression stream for the residual-shape gate:
    # A -> learnable B -> concentrated FAULT -> B recall -> novel/OOD.
    cfg_ch = X_test.shape[-1]
    base_series = synth_multivariate(4200, cfg_ch, seed=0)
    Xs, Ys, meta = build_stream(
        base_series, window=WINDOW, stride=STRIDE, seg_len=600, seed=0
    )
    first_end = meta.segments[0]["win_end"]
    train_X = Xs[:first_end][Ys[:first_end] == 0]
    print("stream source: scripted synthetic gate regression")
else:
    raise ValueError(f"unknown STREAM_MODE: {STREAM_MODE}")

print("stream windows:", Xs.shape, "anomaly windows:", int(Ys.sum()))
for s in meta.segments:
    print(f"  [{s['win_start']:4d}..{s['win_end']:4d}] {s['name']} ({s['kind']})")


In [ ]:
# 7) Train the base model on the clean first regime, then stream.
import random
import numpy as np
import torch
from sage_monitor.config import SAGEConfig
from sage_monitor.controller import SAGEController

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)
requested_device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = SAGEConfig(
    n_channels=Xs.shape[-1], window=WINDOW,
    train_epochs=6, device=requested_device, gate_min_obs=32,
)
print("requested device:", requested_device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

assert train_X is not None and len(train_X) > 0

ctrl = SAGEController(cfg)
ctrl.train_base(train_X, verbose=True)
if ctrl.anomaly_head._ch_sd is not None:
    ch_sd = ctrl.anomaly_head._ch_sd.detach().float().cpu()
    print("calibrated residual scale:",
          f"min={ch_sd.min().item():.6g}",
          f"median={ch_sd.median().item():.6g}",
          f"max={ch_sd.max().item():.6g}")
records = ctrl.run_stream(Xs, labels=Ys, log_every=150)
print("streaming complete:", len(records), "steps")


In [ ]:
# 8) Full metric report incl. safety metrics
from sage_monitor import metrics as M
import json

report = M.summarise(ctrl, records, meta.segments, Ys.numpy())

print("== detection ==")
for k in ("precision", "recall", "f1"):
    print(f"  {k}: {report['detection'][k]:.3f}")
print("== point-adjust ==")
for k in ("precision", "recall", "f1"):
    print(f"  {k}: {report['detection_point_adjust'][k]:.3f}")
print("  missed:", report["missed"], " delays:", report["delays"])
print("  false alarm rate:", round(report["false_alarm_rate"], 4))
print("== continual / safety ==")
print("  regimes:", report["n_regimes"])
print("  adaptations:", report["stats"]["adaptations"],
      "unsafe:", report["stats"]["unsafe_adaptations"])
print("  recalls:", report["stats"]["recalls"],
      "rejects:", report["stats"]["rejects"],
      "alerts:", report["stats"]["alerts"])
print("  UNSAFE ADAPTATION RATE:", round(report["unsafe_adaptation_rate"], 3))
print("  residual-shape telemetry:",
      "max_zscore present" if records and "max_zscore" in records[0]
      else "max_zscore missing")
print("== segment action / residual-shape checks ==")
from collections import Counter
for s in meta.segments:
    seg_records = records[s["win_start"]:s["win_end"]]
    actions = Counter(r["action"] for r in seg_records)
    max_z = [r["max_zscore"] for r in seg_records]
    exceed = [r["exceed_count"] for r in seg_records]
    if max_z:
        shape = (f"max_z=[{min(max_z):.1f}, {max(max_z):.1f}], "
                 f"exceed=[{min(exceed)}, {max(exceed)}]")
    else:
        shape = "no records"
    print(f"  {s['name']}: {dict(actions)}; {shape}")
print("== segment anomaly scores ==")
for n, v in report["segment_anomaly_score"].items():
    print(f"  {n}: {v:.3f}")

with open("run_report.json", "w") as f:
    json.dump(report, f, indent=2, default=float)
print("saved run_report.json")


In [ ]:
# 9) Visualise the decision stream
import matplotlib.pyplot as plt
import numpy as np

t = np.arange(len(records))
drift = [r["drift"] for r in records]
anom = [r["anomaly"] for r in records]
ood = [r["ood"] for r in records]
act = [1 if r["action"] in ("REJECT", "ALERT") else 0 for r in records]

fig, axs = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
axs[0].plot(t, drift, lw=1, color="tab:blue"); axs[0].set_ylabel("drift")
axs[1].plot(t, anom, lw=1, color="tab:red"); axs[1].set_ylabel("anomaly")
axs[2].plot(t, ood, lw=1, color="tab:purple"); axs[2].set_ylabel("OOD")
axs[3].fill_between(t, act, step="mid", color="tab:red", alpha=0.6)
axs[3].set_ylabel("flag")
for s in meta.segments:
    for ax in axs:
        ax.axvspan(s["win_start"], s["win_end"], alpha=0.05)
    axs[0].text(s["win_start"], 1.0, s["name"], fontsize=7, rotation=90, va="top")
plt.tight_layout()
plt.show()


In [ ]:
# 10) Save the model and download artifacts
ctrl.save("sage_model.pt")
try:
    from google.colab import files
    files.download("run_report.json")
    files.download("sage_model.pt")
except Exception as e:
    print("artifact download skipped:", e)
